# Stream-Stream Join with Watermarking: A Step-by-Step Demo

This notebook demonstrates how **watermarking** and **range conditions** work together in Spark Structured Streaming stream-stream joins.

## Scenario
We have two streams:
- **Ad Impressions** — when an ad is shown to a user
- **Ad Clicks** — when a user clicks on an ad

We want to join clicks to impressions, but only if the click happened **within 10 minutes** of the impression (the range condition).

## Key Concepts

| Concept | What it does |
|---|---|
| **Watermark** | Tells Spark how late data can arrive. Defined as `max(event_time) - threshold`. Data older than the watermark is dropped. |
| **Range condition** | Restricts the join to matching rows within a time window (e.g., click within 10 min of impression). |
| **Together** | The watermark + range condition let Spark know when it's safe to evict old state — without them, state grows unboundedly. |

## What We'll Show
| Run | Scenario | Expected Result |
|---|---|---|
| 1 | On-time data, all within range | All pairs match |
| 2 | Clicks at the edge of the 10-min range | In-range clicks match; out-of-range click is excluded |
| 3 | Late data arriving after the watermark | Dropped — no new matches |
| 4 | Late-ish data still within the watermark | Accepted — produces a match |
| 5 | Advance watermark to trigger state eviction | clk_5 evicted from state store |

## Setup
Create the Delta tables and define paths. Running this cell resets all state so the demo is repeatable.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import time

# --- Paths ---
demo_catalog = "alexn"
demo_schema = "wmdemo"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {demo_catalog}.{demo_schema}")

impressions_table = f"{demo_catalog}.{demo_schema}.impressions"
clicks_table = f"{demo_catalog}.{demo_schema}.clicks"
output_table = f"{demo_catalog}.{demo_schema}.matched_clicks"
checkpoint_path = "/Volumes/alexn/default/v/checkpoints/wmdemo"

# --- Clean up prior runs ---
spark.sql(f"DROP TABLE IF EXISTS {output_table}")
spark.sql(f"DROP TABLE IF EXISTS {impressions_table}")
spark.sql(f"DROP TABLE IF EXISTS {clicks_table}")
dbutils.fs.rm(checkpoint_path, recurse=True)

# --- Create empty Delta tables with schema ---
impressions_schema = StructType([
    StructField("impression_id", StringType()),
    StructField("ad_id", StringType()),
    StructField("impression_time", TimestampType()),
])

clicks_schema = StructType([
    StructField("click_id", StringType()),
    StructField("ad_id", StringType()),
    StructField("click_time", TimestampType()),
])

spark.createDataFrame([], impressions_schema).write.format("delta").saveAsTable(impressions_table)
spark.createDataFrame([], clicks_schema).write.format("delta").saveAsTable(clicks_table)

print(f"Schema:      {demo_catalog}.{demo_schema}")
print(f"Impressions: {impressions_table}")
print(f"Clicks:      {clicks_table}")
print(f"Output:      {output_table}")
print(f"Checkpoint:  {checkpoint_path}")
print("Setup complete.")

Schema:      alexn.wmdemo
Impressions: alexn.wmdemo.impressions
Clicks:      alexn.wmdemo.clicks
Output:      alexn.wmdemo.matched_clicks
Checkpoint:  /Volumes/alexn/default/v/checkpoints/wmdemo
Setup complete.


## Helper Functions

In [0]:
from datetime import datetime

def ts(time_str):
    """Shorthand to create a timestamp on 2025-01-15."""
    return datetime.strptime(f"2025-01-15 {time_str}", "%Y-%m-%d %H:%M")

def write_impressions(rows):
    """Append impression rows to the Delta table.
    rows: list of (impression_id, ad_id, time_str) tuples.
    """
    data = [(r[0], r[1], ts(r[2])) for r in rows]
    df = spark.createDataFrame(data, impressions_schema)
    df.write.format("delta").mode("append").saveAsTable(impressions_table)
    print(f"Wrote {len(rows)} impressions:")
    for r in rows:
        print(f"  {r[0]:8s}  ad={r[1]:5s}  time={r[2]}")

def write_clicks(rows):
    """Append click rows to the Delta table.
    rows: list of (click_id, ad_id, time_str) tuples.
    """
    data = [(r[0], r[1], ts(r[2])) for r in rows]
    df = spark.createDataFrame(data, clicks_schema)
    df.write.format("delta").mode("append").saveAsTable(clicks_table)
    print(f"Wrote {len(rows)} clicks:")
    for r in rows:
        print(f"  {r[0]:8s}  ad={r[1]:5s}  time={r[2]}")

# define listener to capture watermark metrics
from pyspark.sql.streaming import StreamingQueryListener

class WatermarkListener(StreamingQueryListener):
    def onQueryProgress(self, event):
        progress = event.progress
        event_time = progress.eventTime
        if event_time:
            watermark = event_time.get("watermark", None)
            if watermark:
                print(f"  eventTime.watermark: {watermark}")
        for op in progress.stateOperators:
            print(f"  stateOp: {op.operatorName} | rows_updated={op.numRowsUpdated}, rows_removed={op.numRowsRemoved}, dropped_by_watermark={op.numRowsDroppedByWatermark}, rows_total={op.numRowsTotal}")

    def onQueryStarted(self, event):
        print(f"Query started: {event.id}")

    def onQueryTerminated(self, event):
        print(f"Query terminated: {event.id}")

spark.streams.addListener(WatermarkListener())

def run_streaming_join():
    """Start the streaming join with watermarks and range condition.
    Uses availableNow trigger so it processes all pending data then stops.
    Prints watermark metrics from StreamingQueryListener.
    """
    impressions_stream = (
        spark.readStream
        .format("delta")
        .table(impressions_table)
        .withWatermark("impression_time", "5 minutes")
    )

    clicks_stream = (
        spark.readStream
        .format("delta")
        .table(clicks_table)
        .withWatermark("click_time", "5 minutes")
    )

    # Join condition: same ad_id AND click happened within 10 minutes of impression
    join_condition = (
        (impressions_stream.ad_id == clicks_stream.ad_id) &
        (clicks_stream.click_time >= impressions_stream.impression_time) &
        (clicks_stream.click_time <= impressions_stream.impression_time + F.expr("INTERVAL 10 MINUTES"))
    )

    joined = impressions_stream.join(clicks_stream, join_condition, "inner")

    query = (
        joined
        .select(
            impressions_stream.impression_id,
            clicks_stream.click_id,
            impressions_stream.ad_id,
            impressions_stream.impression_time,
            clicks_stream.click_time,
            (F.unix_timestamp("click_time") - F.unix_timestamp("impression_time")).cast("int").alias("delay_seconds"),
        )
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(output_table)
    )

    query.awaitTermination()
    print("Streaming join complete.")

def show_results():
    """Display all matched click-impression pairs so far."""
    display(
        spark.table(output_table)
        .orderBy("impression_time", "click_time")
    )

def show_global_watermarks():
    import json
    from datetime import datetime, timezone

    # Read watermark from each commit log in the checkpoint
    commits_path = f"/Volumes/alexn/default/v/checkpoints/wmdemo/commits"
    commit_files = [f for f in dbutils.fs.ls(commits_path) if f.name.strip('/').isdigit()]

    rows = []
    for f in sorted(commit_files, key=lambda x: int(x.name)):
        content = dbutils.fs.head(f.path)
        # Commit files have a version line (e.g. "v1") followed by JSON
        lines = content.strip().split('\n')
        meta = json.loads(lines[-1])
        wm_ms = meta.get("nextBatchWatermarkMs", None)
        if wm_ms is not None:
            wm_ts = datetime.fromtimestamp(wm_ms / 1000, tz=timezone.utc)
            rows.append((int(f.name), wm_ms, str(wm_ts)))

    display(spark.createDataFrame(rows, ["batch_id", "watermark_epoch_ms", "watermark_timestamp"]))

---
## Run 1: On-Time Data — Everything Matches

We write 3 impressions and 3 corresponding clicks. Every click happens within a few minutes of its impression — well within the **10-minute range** and the **5-minute watermark**.

| Impression | Time | Click | Time | Delay |
|---|---|---|---|---|
| imp_1 (ad_A) | 12:00 | clk_1 (ad_A) | 12:03 | 3 min |
| imp_2 (ad_B) | 12:02 | clk_2 (ad_B) | 12:06 | 4 min |
| imp_3 (ad_C) | 12:05 | clk_3 (ad_C) | 12:08 | 3 min |

**Expected: all 3 pairs match.**

In [0]:
write_impressions([
    ("imp_1", "ad_A", "12:00"),
    ("imp_2", "ad_B", "12:02"),
    ("imp_3", "ad_C", "12:05"),
])

write_clicks([
    ("clk_1", "ad_A", "12:03"),
    ("clk_2", "ad_B", "12:06"),
    ("clk_3", "ad_C", "12:08"),
])

run_streaming_join()

Wrote 3 impressions:
  imp_1     ad=ad_A   time=12:00
  imp_2     ad=ad_B   time=12:02
  imp_3     ad=ad_C   time=12:05
Wrote 3 clicks:
  clk_1     ad=ad_A   time=12:03
  clk_2     ad=ad_B   time=12:06
  clk_3     ad=ad_C   time=12:08
Query started: f4fa1079-ad40-425e-8067-428e28ab54a9
  eventTime.watermark: 1970-01-01T00:00:00.000Z
  stateOp: symmetricHashJoin | rows_updated=6, rows_removed=0, dropped_by_watermark=0, rows_total=6
  eventTime.watermark: 2025-01-15T12:00:00.000Z
  stateOp: symmetricHashJoin | rows_updated=0, rows_removed=0, dropped_by_watermark=0, rows_total=6
Streaming join complete.
Query terminated: f4fa1079-ad40-425e-8067-428e28ab54a9


NOTE: Two events (microbatches) are shown because availableNow=True trigger always ends with an extra empty ("no-data") batch.


[source](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)

### Run 1 Results
All 3 pairs should appear. After this batch:
- **Impression watermark** = max(12:05) − 5 min = **12:00**
- **Click watermark** = max(12:08) − 5 min = **12:03**

In [0]:
show_results()

impression_id,click_id,ad_id,impression_time,click_time,delay_seconds
imp_1,clk_1,ad_A,2025-01-15T12:00:00Z,2025-01-15T12:03:00Z,180
imp_2,clk_2,ad_B,2025-01-15T12:02:00Z,2025-01-15T12:06:00Z,240
imp_3,clk_3,ad_C,2025-01-15T12:05:00Z,2025-01-15T12:08:00Z,180


In [0]:
spark.read.format("statestore").option("joinSide", "left").load(checkpoint_path).display()

key,value,partition_id
List(ad_A),"List(imp_1, ad_A, 2025-01-15T12:00:00Z)",90
List(ad_B),"List(imp_2, ad_B, 2025-01-15T12:02:00Z)",143
List(ad_C),"List(imp_3, ad_C, 2025-01-15T12:05:00Z)",197


In [0]:
spark.read.format("statestore").option("joinSide", "right").load(checkpoint_path).display()

key,value,partition_id
List(ad_A),"List(clk_1, ad_A, 2025-01-15T12:03:00Z)",90
List(ad_B),"List(clk_2, ad_B, 2025-01-15T12:06:00Z)",143
List(ad_C),"List(clk_3, ad_C, 2025-01-15T12:08:00Z)",197


In [0]:
show_global_watermarks()

batch_id,watermark_epoch_ms,watermark_timestamp
0,1736942400000,2025-01-15 12:00:00+00:00
1,1736942400000,2025-01-15 12:00:00+00:00


---
## Run 2: Range Condition Boundary

Now we test the **10-minute range condition**. We write 2 new impressions and 3 clicks:

| Impression | Time | Click | Time | Delay | Within 10-min range? |
|---|---|---|---|---|---|
| imp_4 (ad_D) | 12:20 | clk_4 (ad_D) | 12:29 | 9 min | **Yes** |
| imp_4 (ad_D) | 12:20 | clk_5 (ad_D) | 12:31 | 11 min | **No — too late** |
| imp_5 (ad_E) | 12:22 | clk_6 (ad_E) | 12:25 | 3 min | **Yes** |

`clk_5` is for the same ad as `imp_4`, but it arrives **11 minutes** after the impression — outside the range.

**Expected: 2 new matches (clk_4 + clk_6). clk_5 is excluded by the range condition. 
It does not show up in the output (but is retained in the state store for now)**

In [0]:
write_impressions([
    ("imp_4", "ad_D", "12:20"),
    ("imp_5", "ad_E", "12:22"),
])

write_clicks([
    ("clk_4", "ad_D", "12:29"),
    ("clk_5", "ad_D", "12:31"),
    ("clk_6", "ad_E", "12:25"),
])

run_streaming_join()

Wrote 2 impressions:
  imp_4     ad=ad_D   time=12:20
  imp_5     ad=ad_E   time=12:22
Wrote 3 clicks:
  clk_4     ad=ad_D   time=12:29
  clk_5     ad=ad_D   time=12:31
  clk_6     ad=ad_E   time=12:25
Query started: f4fa1079-ad40-425e-8067-428e28ab54a9
  eventTime.watermark: 2025-01-15T12:00:00.000Z
  stateOp: symmetricHashJoin | rows_updated=5, rows_removed=0, dropped_by_watermark=0, rows_total=11
  eventTime.watermark: 2025-01-15T12:17:00.000Z
  stateOp: symmetricHashJoin | rows_updated=0, rows_removed=6, dropped_by_watermark=0, rows_total=5
Query terminated: f4fa1079-ad40-425e-8067-428e28ab54a9
Streaming join complete.


NOTE: 6 rows are evicted from the state store because they can no longer participate in future joins based on the combination of watermark and range condition.``

### Run 2 Results
You should see **5 total rows** (3 from Run 1 + 2 new). `clk_5` at 12:31 is absent — it was 11 minutes after `imp_4`, exceeding the 10-minute range.

Updated watermarks:
- **Impression watermark** = max(12:22) − 5 min = **12:17**
- **Click watermark** = max(12:31) − 5 min = **12:26**

In [0]:
show_results()

impression_id,click_id,ad_id,impression_time,click_time,delay_seconds
imp_1,clk_1,ad_A,2025-01-15T12:00:00Z,2025-01-15T12:03:00Z,180
imp_2,clk_2,ad_B,2025-01-15T12:02:00Z,2025-01-15T12:06:00Z,240
imp_3,clk_3,ad_C,2025-01-15T12:05:00Z,2025-01-15T12:08:00Z,180
imp_4,clk_4,ad_D,2025-01-15T12:20:00Z,2025-01-15T12:29:00Z,540
imp_5,clk_6,ad_E,2025-01-15T12:22:00Z,2025-01-15T12:25:00Z,180


In [0]:
spark.read.format("statestore").option("joinSide", "left").load(checkpoint_path).display()

key,value,partition_id
List(ad_D),"List(imp_4, ad_D, 2025-01-15T12:20:00Z)",5
List(ad_E),"List(imp_5, ad_E, 2025-01-15T12:22:00Z)",67


In [0]:
spark.read.format("statestore").option("joinSide", "right").load(checkpoint_path).display()

key,value,partition_id
List(ad_D),"List(clk_4, ad_D, 2025-01-15T12:29:00Z)",5
List(ad_D),"List(clk_5, ad_D, 2025-01-15T12:31:00Z)",5
List(ad_E),"List(clk_6, ad_E, 2025-01-15T12:25:00Z)",67


In [0]:
show_global_watermarks()

batch_id,watermark_epoch_ms,watermark_timestamp
0,1736942400000,2025-01-15 12:00:00+00:00
1,1736942400000,2025-01-15 12:00:00+00:00
2,1736943420000,2025-01-15 12:17:00+00:00
3,1736943420000,2025-01-15 12:17:00+00:00


---
## Run 3: Late Data — Dropped by Watermark

This is the key watermarking test. We write an impression at **12:10** and a click at **12:12** — data that "arrived late."

But the watermarks have already advanced past these times:
- Impression watermark is **12:17** → an impression at 12:10 is **behind** the watermark
- Click watermark is **12:26** → a click at 12:12 is **behind** the watermark

Spark drops both records because they are too late.

**Expected: no new matches. The output table stays at 5 rows.**

In [0]:
write_impressions([
    ("imp_6", "ad_F", "12:10"),
])

write_clicks([
    ("clk_7", "ad_F", "12:12"),
])

run_streaming_join()

Wrote 1 impressions:
  imp_6     ad=ad_F   time=12:10
Wrote 1 clicks:
  clk_7     ad=ad_F   time=12:12
Query started: f4fa1079-ad40-425e-8067-428e28ab54a9
  eventTime.watermark: 2025-01-15T12:17:00.000Z
  stateOp: symmetricHashJoin | rows_updated=0, rows_removed=0, dropped_by_watermark=2, rows_total=5
Query terminated: f4fa1079-ad40-425e-8067-428e28ab54a9
Streaming join complete.


### Run 3 Results
Still **5 rows** — the late data was silently dropped. This is the watermark in action: it protects the system from unbounded state growth by discarding data that arrives too late to matter.

In [0]:
show_results()

impression_id,click_id,ad_id,impression_time,click_time,delay_seconds
imp_1,clk_1,ad_A,2025-01-15T12:00:00Z,2025-01-15T12:03:00Z,180
imp_2,clk_2,ad_B,2025-01-15T12:02:00Z,2025-01-15T12:06:00Z,240
imp_3,clk_3,ad_C,2025-01-15T12:05:00Z,2025-01-15T12:08:00Z,180
imp_4,clk_4,ad_D,2025-01-15T12:20:00Z,2025-01-15T12:29:00Z,540
imp_5,clk_6,ad_E,2025-01-15T12:22:00Z,2025-01-15T12:25:00Z,180


---
## Run 4: Late-ish Data — Still Within Watermark

Not all "late" data gets dropped. We write an impression at **12:21** and a click at **12:27**.

Checking against the watermarks:
- Impression at 12:21 > impression watermark of **12:17** → **accepted**
- Click at 12:27 > click watermark of **12:26** → **accepted**
- Delay = 5 minutes → **within the 10-minute range**

Even though this data arrives in a later micro-batch than Run 2, its event times are still within the watermark threshold, so Spark processes it normally.

**Expected: 1 new match → 6 total rows.**

In [0]:
write_impressions([
    ("imp_7", "ad_G", "12:21"),
])

write_clicks([
    ("clk_8", "ad_G", "12:27"),
])

run_streaming_join()

Wrote 1 impressions:
  imp_7     ad=ad_G   time=12:21
Wrote 1 clicks:
  clk_8     ad=ad_G   time=12:27
Query started: f4fa1079-ad40-425e-8067-428e28ab54a9
  eventTime.watermark: 2025-01-15T12:17:00.000Z
  stateOp: symmetricHashJoin | rows_updated=2, rows_removed=0, dropped_by_watermark=0, rows_total=7
Query terminated: f4fa1079-ad40-425e-8067-428e28ab54a9
Streaming join complete.


### Run 4 Results
**6 rows** — the new pair (imp_7, clk_8) matched successfully.

In [0]:
show_results()

impression_id,click_id,ad_id,impression_time,click_time,delay_seconds
imp_1,clk_1,ad_A,2025-01-15T12:00:00Z,2025-01-15T12:03:00Z,180
imp_2,clk_2,ad_B,2025-01-15T12:02:00Z,2025-01-15T12:06:00Z,240
imp_3,clk_3,ad_C,2025-01-15T12:05:00Z,2025-01-15T12:08:00Z,180
imp_4,clk_4,ad_D,2025-01-15T12:20:00Z,2025-01-15T12:29:00Z,540
imp_7,clk_8,ad_G,2025-01-15T12:21:00Z,2025-01-15T12:27:00Z,360
imp_5,clk_6,ad_E,2025-01-15T12:22:00Z,2025-01-15T12:25:00Z,180


In [0]:
spark.read.format("statestore").option("joinSide", "left").load(checkpoint_path).display()

key,value,partition_id
List(ad_D),"List(imp_4, ad_D, 2025-01-15T12:20:00Z)",5
List(ad_G),"List(imp_7, ad_G, 2025-01-15T12:21:00Z)",16
List(ad_E),"List(imp_5, ad_E, 2025-01-15T12:22:00Z)",67


In [0]:
spark.read.format("statestore").option("joinSide", "right").load(checkpoint_path).display()

key,value,partition_id
List(ad_D),"List(clk_4, ad_D, 2025-01-15T12:29:00Z)",5
List(ad_D),"List(clk_5, ad_D, 2025-01-15T12:31:00Z)",5
List(ad_G),"List(clk_8, ad_G, 2025-01-15T12:27:00Z)",16
List(ad_E),"List(clk_6, ad_E, 2025-01-15T12:25:00Z)",67


---
## Run 5: State Eviction — clk_5 Finally Removed

Remember `clk_5` (ad_D, click_time=12:31) from Run 2? It didn't match any impression (11 min > 10-min range), but Spark kept it in the right-side state store — because a future impression with `impression_time` between 12:21 and 12:31 could still arrive and match it.

Now we advance the impression watermark past 12:31 so that's no longer possible. We write an impression at **12:40**:
- **Impression watermark** advances to max(12:40) − 5 min = **12:35**
- Since 12:35 > 12:31, no future impression can have `impression_time ≤ 12:31`
- Therefore `clk_5` can never be matched → Spark **evicts it** from state

We also write a click at 12:43 for the new impression (3-min delay, within range) so we get a match.

**Expected: 1 new match (imp_8 + clk_9) → 7 total rows. clk_5 disappears from the right-side state store.**

In [0]:
write_impressions([
    ("imp_8", "ad_H", "12:40"),
])

write_clicks([
    ("clk_9", "ad_H", "12:43"),
])

run_streaming_join()

Wrote 1 impressions:
  imp_8     ad=ad_H   time=12:40
Wrote 1 clicks:
  clk_9     ad=ad_H   time=12:43
Query started: f4fa1079-ad40-425e-8067-428e28ab54a9
  eventTime.watermark: 2025-01-15T12:17:00.000Z
  stateOp: symmetricHashJoin | rows_updated=2, rows_removed=0, dropped_by_watermark=0, rows_total=9
  eventTime.watermark: 2025-01-15T12:35:00.000Z
  stateOp: symmetricHashJoin | rows_updated=0, rows_removed=7, dropped_by_watermark=0, rows_total=2
Query terminated: f4fa1079-ad40-425e-8067-428e28ab54a9
Streaming join complete.


### Run 5 Results
**7 rows** — the new pair (imp_8, clk_9) matched. Check the right-side state store below — `clk_5` should be gone.

Updated watermarks:
- **Impression watermark** = max(12:40) − 5 min = **12:35**
- **Click watermark** = max(12:43) − 5 min = **12:38**

In [0]:
show_results()

impression_id,click_id,ad_id,impression_time,click_time,delay_seconds
imp_1,clk_1,ad_A,2025-01-15T12:00:00Z,2025-01-15T12:03:00Z,180
imp_2,clk_2,ad_B,2025-01-15T12:02:00Z,2025-01-15T12:06:00Z,240
imp_3,clk_3,ad_C,2025-01-15T12:05:00Z,2025-01-15T12:08:00Z,180
imp_4,clk_4,ad_D,2025-01-15T12:20:00Z,2025-01-15T12:29:00Z,540
imp_7,clk_8,ad_G,2025-01-15T12:21:00Z,2025-01-15T12:27:00Z,360
imp_5,clk_6,ad_E,2025-01-15T12:22:00Z,2025-01-15T12:25:00Z,180
imp_8,clk_9,ad_H,2025-01-15T12:40:00Z,2025-01-15T12:43:00Z,180


In [0]:
spark.read.format("statestore").option("joinSide", "left").load(checkpoint_path).display()

key,value,partition_id
List(ad_H),"List(imp_8, ad_H, 2025-01-15T12:40:00Z)",116


In the right-side state store below, `clk_5` (ad_D, 12:31) is **no longer present** — it was evicted because the impression watermark (12:35) now guarantees no future impression can match it.

In [0]:
spark.read.format("statestore").option("joinSide", "right").load(checkpoint_path).display()

key,value,partition_id
List(ad_H),"List(clk_9, ad_H, 2025-01-15T12:43:00Z)",116


In [0]:
show_global_watermarks()

batch_id,watermark_epoch_ms,watermark_timestamp
0,1736942400000,2025-01-15 12:00:00+00:00
1,1736942400000,2025-01-15 12:00:00+00:00
2,1736943420000,2025-01-15 12:17:00+00:00
3,1736943420000,2025-01-15 12:17:00+00:00
4,1736943420000,2025-01-15 12:17:00+00:00
5,1736943420000,2025-01-15 12:17:00+00:00
6,1736944500000,2025-01-15 12:35:00+00:00
7,1736944500000,2025-01-15 12:35:00+00:00


---
## Summary

| Run | What Happened | Matches? | Why |
|---|---|---|---|
| 1 | On-time impressions + clicks, all within range | **3 matches** | All data within watermark and 10-min range |
| 2 | Clicks at 9 min (in range) and 11 min (out of range) | **2 new matches** | Range condition filters out the 11-min click |
| 3 | Late impression at 12:10 + click at 12:12 | **0 new matches** | Both behind the watermark → dropped |
| 4 | Impression at 12:21 + click at 12:27 | **1 new match** | Event times still ahead of watermark → accepted |
| 5 | Impression at 12:40 advances watermark to 12:35 | **1 new match** | clk_5 evicted — impression watermark (12:35) > clk_5 time (12:31) |

### Key Takeaways

1. **Watermarks control late data.** Data with event times behind the watermark (`max_event_time - delay`) is dropped before entering state. This prevents stale data from accumulating.

2. **The range condition controls match validity.** Even if data passes the watermark, it must satisfy the time range to produce a join result.

3. **Both are needed to bound state in a join, but for different reasons.** The watermark alone filters late *input*, but cannot evict rows already *in* state — without a range condition, any future row could match any past row, so nothing is safe to remove. The range condition gives Spark the eviction rule: once `watermark > row_time + range`, that buffered row can never be matched and is evicted. Together, they bound both what enters state and how long it stays.
[info on eviction requirements in streaming joins](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html#inner-joins-with-optional-watermarking)

4. **The watermark is not a wall clock — it's driven by the data.** It advances based on the maximum event time seen, not the current time. This makes it testable and deterministic.

## Cleanup
Uncomment and run the cell below to remove all demo tables and state.

In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {output_table}")
# spark.sql(f"DROP TABLE IF EXISTS {impressions_table}")
# spark.sql(f"DROP TABLE IF EXISTS {clicks_table}")
# spark.sql(f"DROP SCHEMA IF EXISTS {demo_catalog}.{demo_schema} CASCADE")
# dbutils.fs.rm(checkpoint_path, recurse=True)
# print("Cleanup complete.")